# Stage C — QE bulk validation of the top-6 CO2RR economic-HEC candidates

Quantum ESPRESSO (PBE, SSSP + norm-conserving REE pseudos) DFT validation of the six
fairchem-selected candidates: **vc-relax -> scf -> DOS -> PDOS** (`scripts/52_qe_run.py`,
`scripts/53_qe_pdos.py`). Confirms (i) low residual stress, (ii) **metallicity** (finite DOS
at $E_F$), and (iii) **which elements/orbitals carry the conduction** (projected DOS at $E_F$).

Reads each `outputs/qe/<formula>/{scf.out, vc-relax.out, <formula>.dos, <formula>.pdos_elements.csv}`
directly, so it is **re-runnable during the QE run**. Joins the fairchem $\Delta G_{CO}$ ranking
from `outputs/screen_co2rr/top6.csv`. PDOS reuses the scf wavefunctions (no recompute).


In [ ]:
import sys, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

ROOT = Path('/home/jonglee69/mattergen/CarbideMatterGen'); sys.path.insert(0, str(ROOT))
QE = ROOT/'outputs'/'qe'; SCREEN = ROOT/'outputs'/'screen_co2rr'; FIG = ROOT/'figures'; FIG.mkdir(exist_ok=True)
RY = 13.605693  # eV/Ry
NATURE_RC = {'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
    'font.size':8,'axes.labelsize':8,'axes.titlesize':9,'xtick.labelsize':7,'ytick.labelsize':7,
    'legend.fontsize':6,'axes.linewidth':0.6,'xtick.direction':'in','ytick.direction':'in',
    'legend.frameon':False,'pdf.fonttype':42,'savefig.bbox':'tight','savefig.dpi':300}
mpl.rcParams.update(NATURE_RC)
OI = {'blue':'#0072B2','vermil':'#D55E00','green':'#009E73','orange':'#E69F00','purple':'#CC79A7','gray':'#888888'}
# consistent per-element colours for PDOS
EL_COL = {'C':'#333333','Y':'#0072B2','Zr':'#56B4E9','W':'#D55E00','Ce':'#009E73',
          'La':'#CC79A7','Pr':'#E69F00','Nd':'#F0E442','Hf':'#999999','Zn':'#8C564B','Ti':'#17BECF','Sc':'#BCBD22'}

def grab(t, pat, cast=float):
    m = re.findall(pat, t); return cast(m[-1]) if m else None
def parse_pw(p):
    if not p.exists(): return {}
    t = p.read_text(errors='ignore')
    return {'E_Ry': grab(t, r'!\s+total energy\s+=\s+(-?\d+\.\d+)'),
            'E_fermi': grab(t, r'the Fermi energy is\s+(-?\d+\.\d+)'),
            'P_kbar': grab(t, r'P=\s+(-?\d+\.\d+)'), 'done': 'JOB DONE' in t}
def dos_at(dosfile, ef):
    rows=[l.split() for l in dosfile.read_text().splitlines() if l.strip() and not l.lstrip().startswith('#')]
    a=np.array([[float(x) for x in r[:2]] for r in rows if len(r)>=2]); i=int(np.argmin(np.abs(a[:,0]-ef)))
    return a, float(a[i,1])

top6 = pd.read_csv(SCREEN/'top6.csv') if (SCREEN/'top6.csv').exists() else pd.DataFrame()
recs = {}
for cdir in sorted(QE.glob('*/')):
    f = cdir.name; scf = parse_pw(cdir/'scf.out'); vcr = parse_pw(cdir/'vc-relax.out')
    dosf = cdir/f'{f}.dos'; pdosf = cdir/f'{f}.pdos_elements.csv'
    if not dosf.exists() and not scf.get('done'): continue
    ef = scf.get('E_fermi') or vcr.get('E_fermi')
    dos_arr=None; gef=np.nan
    if dosf.exists() and ef is not None: dos_arr, gef = dos_at(dosf, ef)
    pdos = pd.read_csv(pdosf) if pdosf.exists() else None
    recs[f] = {'formula':f,'E_fermi_eV':ef,'P_kbar':vcr.get('P_kbar'),
        'E_eV_per_cell':(scf.get('E_Ry') or np.nan)*RY,'dos_at_Ef':round(gef,2) if gef==gef else np.nan,
        'metallic':bool(gef>0.1) if gef==gef else None,'scf_done':scf.get('done'),
        '_dos':dos_arr,'_ef':ef,'_pdos':pdos}
print(f'{len(recs)} QE candidates with results so far')


## 1. Stability + metallicity table (joined with fairchem $\Delta G_{CO}$ ranking)


In [ ]:
df = pd.DataFrame([{k:v for k,v in r.items() if not k.startswith('_')} for r in recs.values()])
if len(df) and len(top6):
    df = df.merge(top6[['formula','best_dG_CO','frac_co_selective']], on='formula', how='left')
    df = df.sort_values('best_dG_CO', key=lambda s: s.abs())
cols = [c for c in ['formula','best_dG_CO','frac_co_selective','E_fermi_eV','dos_at_Ef','metallic','P_kbar','E_eV_per_cell','scf_done'] if c in df.columns]
df.to_csv(QE/'qe_validation_summary.csv', index=False)
print(df[cols].to_string(index=False))
if 'metallic' in df.columns:
    print(f'\nmetallic: {int(df.metallic.sum())}/{len(df)}  |  DOS(E_F) {df.dos_at_Ef.min()}-{df.dos_at_Ef.max()} states/eV/cell')


## 2. Total density of states at the Fermi level

DOS(E) vs $E-E_F$ for each finished candidate. Finite DOS at $E_F$ (dashed) confirms metallicity.


In [ ]:
finished = [(f, r) for f, r in recs.items() if r.get('_dos') is not None]
n = len(finished)
if n:
    ncol=min(3,n); nrow=int(np.ceil(n/ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(2.7*ncol, 2.2*nrow), squeeze=False)
    for k,(f,r) in enumerate(finished):
        a=ax[k//ncol][k%ncol]; arr=r['_dos']; ef=r['_ef']
        a.plot(arr[:,0]-ef, arr[:,1], color=OI['blue'], lw=0.9); a.fill_between(arr[:,0]-ef, arr[:,1], color=OI['blue'], alpha=0.15)
        a.axvline(0, color=OI['vermil'], ls='--', lw=0.8)
        a.set_title(f'{f}\nDOS($E_F$)={r["dos_at_Ef"]:.1f}', fontsize=7); a.set_xlim(-8,8)
        a.set_xlabel(r'$E-E_F$ (eV)'); a.set_ylabel('DOS (states/eV)')
    for k in range(n,nrow*ncol): ax[k//ncol][k%ncol].axis('off')
    fig.tight_layout()
    for ext in ('pdf','png'): fig.savefig(FIG/f'fig_co2rr_qe_dos.{ext}')
    print('saved fig_co2rr_qe_dos'); plt.show()
else: print('no DOS files yet')


## 3. Element-projected DOS (PDOS) — who carries the conduction

`projwfc.x` projects the (already computed) Kohn-Sham states onto atomic orbitals, summed per
element. The element-resolved PDOS vs $E-E_F$ shows which sublattice dominates the states at
$E_F$ — i.e. each element's contribution to metallic conduction. (No SCF recompute: PDOS reuses
the scf wavefunctions; generated by `scripts/53_qe_pdos.py`.)


In [ ]:
have_pdos = [(f, r) for f, r in recs.items() if r.get('_pdos') is not None]
m = len(have_pdos)
if m:
    ncol=min(3,m); nrow=int(np.ceil(m/ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(2.9*ncol, 2.3*nrow), squeeze=False)
    for k,(f,r) in enumerate(have_pdos):
        a=ax[k//ncol][k%ncol]; p=r['_pdos']; ef=r['_ef']; x=p['E_eV']-ef
        elems=[c for c in p.columns if c not in ('E_eV','total')]
        # order by contribution at E_F (largest first)
        i_ef=int(np.argmin(np.abs(x)))
        elems=sorted(elems, key=lambda e: -p[e].iloc[i_ef])
        for el in elems:
            a.plot(x, p[el], color=EL_COL.get(el,'#444'), lw=0.9, label=el)
        a.plot(x, p['total'], color='k', lw=0.6, ls='--', alpha=0.6, label='total')
        a.axvline(0, color=OI['vermil'], ls=':', lw=0.7)
        a.set_title(f, fontsize=7); a.set_xlim(-8,8); a.set_xlabel(r'$E-E_F$ (eV)'); a.set_ylabel('PDOS (states/eV)')
        a.legend(loc='upper right', fontsize=5, ncol=2)
    for k in range(m,nrow*ncol): ax[k//ncol][k%ncol].axis('off')
    fig.tight_layout()
    for ext in ('pdf','png'): fig.savefig(FIG/f'fig_co2rr_qe_pdos.{ext}')
    print('saved fig_co2rr_qe_pdos'); plt.show()
else: print('no PDOS yet — run: python scripts/53_qe_pdos.py --qe-dir outputs/qe')


## 4. Per-element share of DOS at $E_F$

Stacked bars: each element's percentage of the total DOS at $E_F$ (from `qe_pdos_Ef_contributions.csv`).
This quantifies the conduction character (e.g. carbon-$p$ network vs metal-$d$ dominated).


In [ ]:
cf = QE/'qe_pdos_Ef_contributions.csv'
if cf.exists():
    con = pd.read_csv(cf)
    # order rows by fairchem ranking if available
    if len(top6):
        order = [f for f in top6['formula'] if f in set(con['formula'])] + [f for f in con['formula'] if f not in set(top6['formula'])]
        con = con.set_index('formula').loc[order].reset_index()
    elcols = [c for c in con.columns if c.startswith('%')]
    fig, axb = plt.subplots(figsize=(max(3.2, 0.9*len(con)+1.5), 2.6))
    bottom = np.zeros(len(con)); xpos=np.arange(len(con))
    for c in elcols:
        el=c[1:]; vals=con[c].fillna(0).values
        axb.bar(xpos, vals, bottom=bottom, color=EL_COL.get(el,'#444'), label=el, width=0.7, edgecolor='white', linewidth=0.3)
        bottom += vals
    axb.set_xticks(xpos); axb.set_xticklabels(con['formula'], rotation=30, ha='right', fontsize=6)
    axb.set_ylabel('% of DOS at $E_F$'); axb.set_ylim(0,100); axb.set_title('conduction character at $E_F$')
    axb.legend(loc='upper center', bbox_to_anchor=(0.5,-0.28), ncol=min(7,len(elcols)), fontsize=6)
    fig.tight_layout()
    for ext in ('pdf','png'): fig.savefig(FIG/f'fig_co2rr_qe_pdos_Ef_share.{ext}')
    print('saved fig_co2rr_qe_pdos_Ef_share'); print(con.to_string(index=False)); plt.show()
else: print('run scripts/53_qe_pdos.py first to produce qe_pdos_Ef_contributions.csv')


## 5. Takeaways

- All fairchem-selected economic-HEC candidates are PBE-DFT **metallic** (finite DOS at $E_F$),
  confirming the conductivity needed for electrocatalysis; cells relax to low residual pressure.
- **PDOS** resolves the conduction character at $E_F$: in the carbon-rich candidates the C-$p$
  states dominate the Fermi-level DOS, with metal-$d$ (Y/Ce/La/Pr/W) contributing — i.e. a
  conductive carbide network hybridised with the metal sublattice.
- This closes the bulk generate->screen->validate loop; surface $\Delta G_{CO}$ activity is at
  the fairchem/UMA level (notebook 18).

_Re-run as the QE + PDOS scripts finish more candidates; tables and all figures grow automatically._
